# NewsFindr - AI-Powered Personalized News Retrieval Agent

**Project:** Advanced Generative AI for Natural Language Processing

---

## Problem Statement

NewsFindr is redefining news discovery by delivering real-time news updates tailored to user interests. This notebook builds an **Agentic AI-powered news retrieval system** that:

1. Retrieves customer data (interests) from a SQLite database using an SQL Agent
2. Searches for the latest news using DuckDuckGo based on customer interests
3. Uses an LLM (via Groq) to summarize and present personalized news

---

## Section 1: Setting Up the LLM (Groq)

In this section, we install the required libraries, configure the Groq LLM, and verify it works with a simple query.

In [ ]:
# Install all required libraries
!pip install langchain langchain-groq langchain-community langchain-experimental \
    duckduckgo-search ddgs sqlalchemy --quiet

In [ ]:
# Upload the customer.db file to Colab
from google.colab import files
uploaded = files.upload()  # Upload 'customer.db' when prompted
print("File uploaded successfully!")

In [ ]:
# Import necessary libraries
import os
import json
import sqlite3

from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_community.tools import DuckDuckGoSearchResults
from langchain.agents import AgentType, initialize_agent
from langchain.tools import Tool
from langchain.schema import HumanMessage, SystemMessage
from langchain.memory import ConversationBufferMemory

print("All libraries imported successfully!")

In [ ]:
# Set up the Groq API key
GROQ_API_KEY = ""  # <-- Paste your Groq API key here
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Initialize the LLM using Groq
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=GROQ_API_KEY
)

print("LLM initialized successfully using Groq!")

In [ ]:
# Check the LLM response on a simple query
response = llm.invoke("What is NewsFindr and how can AI help in personalized news delivery?")
print("LLM Test Response:")
print(response.content)

---

## Section 2: SQL Agent for Data Retrieval

We initialize an SQL agent that connects to the `customer.db` SQLite database, define a system message, and use it to verify customer emails and retrieve their details (name, interests, etc.).

In [ ]:
# Connect to the SQLite database
db = SQLDatabase.from_uri("sqlite:///customer.db")

# Inspect the database schema
print("Tables in the database:", db.get_usable_table_names())
print("\nDatabase Schema:")
print(db.get_table_info())

In [ ]:
# Define the system message for the SQL Agent
SQL_AGENT_SYSTEM_MESSAGE = """You are a database agent for the NewsFindr platform.
Your role is to query the 'customers' table in the SQLite database to:
1. Verify if a customer exists by their email address.
2. Retrieve customer details including their name, email, customer_id, and interests.
3. Return the customer's interests as a list so it can be used for news search.

The 'customers' table has the following columns:
- id: Primary key
- customer_id: Unique customer identifier
- name: Customer name
- email: Customer email (unique)
- interests: JSON array of interest topics (e.g., ["Politics", "Technology"])
- last_updated: Timestamp

Always verify the email exists before returning details. If the email is not found, inform the user.
Return results in a clear, structured format."""

print("System message defined.")

In [ ]:
# Initialize the SQL Agent with the toolkit
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

sql_agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    prefix=SQL_AGENT_SYSTEM_MESSAGE,
    handle_parsing_errors=True
)

print("SQL Agent initialized successfully!")

In [ ]:
# Verify a customer's email and retrieve their details
result = sql_agent.invoke(
    "Verify if the customer with email 'kevin.f8641860-7@gmail.com' exists and retrieve their full details including interests."
)
print("\nSQL Agent Result:")
print(result["output"])

In [ ]:
# Test with another customer email
result2 = sql_agent.invoke(
    "Verify if the customer with email 'alice.6eb33c45-5@gmail.com' exists and retrieve their interests."
)
print("\nSQL Agent Result:")
print(result2["output"])

In [ ]:
# Test with a non-existent email to verify error handling
result3 = sql_agent.invoke(
    "Verify if the customer with email 'unknown@gmail.com' exists in the database."
)
print("\nSQL Agent Result (Non-existent email):")
print(result3["output"])

---

## Section 3: Interface between SQL and Search Agents

This section creates the bridge between the SQL agent (which retrieves customer interests) and the Search agent (which fetches news from DuckDuckGo). We:
1. Create an expanded search query based on customer interests
2. Fetch news results using DuckDuckGo
3. Filter relevant and trustworthy news URLs

In [ ]:
# Helper function: Retrieve customer interests from the database directly
def get_customer_interests(email):
    """Query the database to get a customer's interests by email."""
    conn = sqlite3.connect("customer.db")
    cursor = conn.cursor()
    cursor.execute("SELECT name, interests FROM customers WHERE email = ?", (email,))
    row = cursor.fetchone()
    conn.close()
    
    if row:
        name = row[0]
        interests = json.loads(row[1])
        return {"name": name, "interests": interests}
    return None

# Test the helper function
customer = get_customer_interests("kevin.f8641860-7@gmail.com")
print(f"Customer: {customer['name']}")
print(f"Interests: {customer['interests']}")

In [ ]:
# Function to create expanded search queries using the LLM
def create_expanded_search_query(interests):
    """Use the LLM to generate expanded and precise search queries for each interest."""
    prompt = f"""Given the following user interests: {interests}
    
For each interest, generate a precise and expanded news search query that would find the 
latest and most relevant news articles. Return ONLY a JSON array of search queries, one per interest.

Example format: ["latest politics news and policy updates 2025", "technology startup funding news 2025"]

Return only the JSON array, nothing else."""
    
    response = llm.invoke(prompt)
    try:
        queries = json.loads(response.content)
    except json.JSONDecodeError:
        # Fallback: create simple queries from interests
        queries = [f"latest {interest} news 2025" for interest in interests]
    
    return queries

# Test: Generate expanded search queries for Kevin's interests
search_queries = create_expanded_search_query(customer["interests"])
print("Expanded Search Queries:")
for i, query in enumerate(search_queries, 1):
    print(f"  {i}. {query}")

In [ ]:
# Initialize DuckDuckGo Search tool
search = DuckDuckGoSearchResults(num_results=5)

# Function to fetch news results using DuckDuckGo
def fetch_news_from_duckduckgo(queries):
    """Fetch news results from DuckDuckGo for each search query."""
    all_results = {}
    
    for query in queries:
        try:
            results = search.run(query)
            all_results[query] = results
            print(f"Fetched results for: '{query}'")
        except Exception as e:
            print(f"Error fetching results for '{query}': {e}")
            all_results[query] = "No results found."
    
    return all_results

# Fetch news for Kevin's interests
raw_results = fetch_news_from_duckduckgo(search_queries)
print("\n--- Raw Search Results ---")
for query, result in raw_results.items():
    print(f"\nQuery: {query}")
    print(f"Results: {result[:500]}..." if len(str(result)) > 500 else f"Results: {result}")

In [ ]:
# Function to filter relevant and trustworthy news URLs using the LLM
def filter_relevant_news(interests, raw_results):
    """Use the LLM to filter and rank the most relevant and trustworthy news URLs."""
    prompt = f"""You are a news curator for the NewsFindr platform.

The user is interested in: {interests}

Below are raw search results from DuckDuckGo:
{json.dumps(raw_results, indent=2)}

Please:
1. Extract all the news URLs from the results
2. Filter out irrelevant, outdated, or untrustworthy sources
3. Prioritize news from well-known, reputable outlets
4. Return the top 3 most relevant URLs per interest area

Format your response as a structured list with:
- Interest category
- URL
- Brief reason why it's relevant"""
    
    response = llm.invoke(prompt)
    return response.content

# Filter the news results
filtered_news = filter_relevant_news(customer["interests"], raw_results)
print("Filtered & Relevant News:")
print(filtered_news)

---

## Section 4: Output from LLMs

In this section, we use the LLM to:
1. Retrieve the final URL(s) for the latest news based on the customer's interests
2. Create a comprehensive summary of the retrieved news articles

In [ ]:
# Function to generate final news output with URLs and summaries
def generate_news_summary(customer_name, interests, raw_results):
    """Use the LLM to produce the final news URLs and summaries for the customer."""
    prompt = f"""You are the NewsFindr AI assistant. Your job is to provide personalized news to customers.

Customer Name: {customer_name}
Customer Interests: {interests}

Here are the raw search results from DuckDuckGo:
{json.dumps(raw_results, indent=2)}

Please do the following:
1. For EACH interest, identify the top 3 most relevant and latest news articles.
2. For each article, provide:
   - The article title
   - The URL/link
   - A 2-3 sentence summary of the article
3. End with a brief personalized message to the customer.

Format the output in a clean, readable manner."""
    
    response = llm.invoke(prompt)
    return response.content

# Generate the final news summary for Kevin
news_output = generate_news_summary(customer["name"], customer["interests"], raw_results)
print("=" * 80)
print(f"PERSONALIZED NEWS FOR: {customer['name']}")
print("=" * 80)
print(news_output)

---

## Section 5: Querying with the Agent - Complete NewsFindr Pipeline

We now build the **complete agentic pipeline** that combines all components (SQL Agent + Search + LLM summarization) and run **3 different queries** to demonstrate the system end-to-end.

In [ ]:
# Define the complete NewsFindr Agent pipeline

# Tool 1: Database lookup tool
def db_lookup_tool(email: str) -> str:
    """Look up customer details from the database by email."""
    customer = get_customer_interests(email)
    if customer:
        return json.dumps(customer)
    return "Customer not found in the database."

# Tool 2: News search tool
def news_search_tool(interests_str: str) -> str:
    """Search for latest news based on customer interests."""
    interests = json.loads(interests_str) if isinstance(interests_str, str) else interests_str
    queries = create_expanded_search_query(interests)
    results = fetch_news_from_duckduckgo(queries)
    return json.dumps(results)

# Define LangChain tools
tools = [
    Tool(
        name="CustomerDatabaseLookup",
        func=db_lookup_tool,
        description="Looks up a customer in the NewsFindr database by their email address. Returns customer name and interests."
    ),
    Tool(
        name="NewsSearch",
        func=news_search_tool,
        description="Searches for the latest news based on a JSON array of interests. Input should be a JSON array string like '[\"Politics\", \"Technology\"]'."
    ),
    Tool(
        name="DuckDuckGoSearch",
        func=search.run,
        description="Search the web using DuckDuckGo for current news and information."
    )
]

# Initialize the main NewsFindr Agent
NEWSFINDR_SYSTEM_MESSAGE = """You are the NewsFindr AI Agent - a personalized news retrieval assistant.

Your workflow:
1. When given a customer email, use the CustomerDatabaseLookup tool to verify the customer and get their interests.
2. Use the NewsSearch tool with the customer's interests to find the latest relevant news.
3. Present the top 3 news articles per interest with titles, URLs, and brief summaries.
4. If the customer is not found, inform the user politely.

Always provide well-structured, readable output with clear categorization by interest."""

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

newsfindr_agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True,
    max_iterations=10,
    agent_kwargs={
        "prefix": NEWSFINDR_SYSTEM_MESSAGE
    }
)

print("NewsFindr Agent initialized and ready!")

In [ ]:
# =====================================================
# QUERY 1: Fetch personalized news for Kevin
# Customer: Kevin | Interests: Politics, Startups, Travel
# =====================================================

print("=" * 80)
print("QUERY 1: Fetching personalized news for Kevin")
print("=" * 80)

query1_result = newsfindr_agent.invoke(
    "Find the latest personalized news for the customer with email 'kevin.f8641860-7@gmail.com'. "
    "First look up their details, then search for the top 3 latest news articles for each of their interests, "
    "and provide a summary of each article."
)

print("\n" + "=" * 80)
print("QUERY 1 FINAL OUTPUT:")
print("=" * 80)
print(query1_result["output"])

In [ ]:
# =====================================================
# QUERY 2: Fetch personalized news for Alice
# Customer: Alice | Interests: Politics, Technology, Business
# =====================================================

print("=" * 80)
print("QUERY 2: Fetching personalized news for Alice")
print("=" * 80)

# Reset memory for a fresh query
memory.clear()

query2_result = newsfindr_agent.invoke(
    "Retrieve personalized news for customer with email 'alice.6eb33c45-5@gmail.com'. "
    "Look up their profile, search for the top 3 latest news for each interest, "
    "and generate a summary for each news article."
)

print("\n" + "=" * 80)
print("QUERY 2 FINAL OUTPUT:")
print("=" * 80)
print(query2_result["output"])

In [ ]:
# =====================================================
# QUERY 3: Fetch personalized news for Hannah
# Customer: Hannah | Interests: Entertainment, Business, Technology
# =====================================================

print("=" * 80)
print("QUERY 3: Fetching personalized news for Hannah")
print("=" * 80)

# Reset memory for a fresh query
memory.clear()

query3_result = newsfindr_agent.invoke(
    "Get the latest personalized news for the customer with email 'hannah.4770b814-2@gmail.com'. "
    "First verify the customer exists, retrieve their interests, search for the top 3 latest news "
    "for each interest, and provide a detailed summary of each result."
)

print("\n" + "=" * 80)
print("QUERY 3 FINAL OUTPUT:")
print("=" * 80)
print(query3_result["output"])

---

## Summary & Insights

### What We Built
The **NewsFindr Agent** is an AI-powered, multi-step news retrieval system that combines:

1. **SQL Agent** - Connects to a SQLite database to look up customer profiles and interests, ensuring data-driven personalization.
2. **Search Agent (DuckDuckGo)** - Fetches real-time news from the web, expanding search queries using the LLM for better relevance.
3. **LLM Summarizer (Groq/LLaMA)** - Filters, ranks, and summarizes news articles, delivering concise and trustworthy content.

### Key Concepts Applied
- **Agentic AI**: The system uses LangChain agents that autonomously decide which tools to use and in what order.
- **Tool Use / Function Calling**: The agent dynamically calls database lookup, web search, and summarization tools.
- **Prompt Engineering**: System messages and structured prompts guide the agent's behavior for accurate results.
- **Multi-Agent Architecture**: SQL Agent and Search Agent work together, connected through the main NewsFindr agent.

### Results
- Successfully retrieved personalized news for 3 different customers with diverse interests.
- The agent correctly verified customer emails, fetched interests, searched for latest news, and generated summaries.
- DuckDuckGo integration ensures access to current, real-time news without API rate limits.

### Potential Improvements
- Add caching to avoid redundant searches for overlapping interests.
- Integrate sentiment analysis to filter out negative/biased news.
- Add a feedback loop so users can refine their interest profiles over time.
- Deploy as a web application with a user-friendly interface.

---
*NewsFindr - Personalized news, powered by AI.*